In [ ]:
!pip install requests


In [ ]:
import requests
import time
import json
from datetime import datetime

top_url = "https://hacker-news.firebaseio.com/v0/topstories.json"
item_url = "https://hacker-news.firebaseio.com/v0/item/{}.json"

headers = {"User-Agent": "TrendPulse/1.0"}

categories = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

def get_category(title):
    title = title.lower()
    for category, keywords in categories.items():
        for word in keywords:
            if word in title:
                return category
    return "others"


try:
    response = requests.get(top_url, headers=headers, timeout=10)
    ids = response.json()[:500]
except Exception as e:
    print("Error getting story IDs:", e)
    ids = []

results = []

count["others"] = 0

# Loop through stories
count = {c: 0 for c in categories}
for sid in ids:
    try:
        r = requests.get(item_url.format(sid), headers=headers, timeout=10)
        story = r.json()

        if not story or "title" not in story:
            continue

        title = story["title"]
        cat = get_category(title)

        # Limit 25 per category
        if count.get(cat, 0) >= 25:
            continue

        info = {
            "post_id": story["id"],
            "title": title,
            "category": cat,
            "score": story.get("score", 0),
            # ✅ Fixed key name
            "num_comments": story.get("descendants", 0),
            "author": story.get("by", ""),
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        results.append(info)
        count[cat] += 1

        # Stop at 124 stories
        if len(results) >= 124:
            break


    except Exception as e:
        print(f"Error fetching story {sid}: {e}")

# Small delay
time.sleep(2)

# Save file
file_name = "trends_" + datetime.now().strftime("%Y%m%d") + ".json"

with open(file_name, "w") as f:
    json.dump(results, f, indent=4)

print("Saved", len(results), "stories to", file_name)

Saved 104 stories to trends_20260406.json
